# polymer_sensitivity_v1 reproducible analysis

## tl;dr

This notebook audits the completed 3×3 OPM Flow parameter sweep and reproduces the core sensitivity comparisons used by the Markdown report.

## Context & Methods

Source files: `outputs/experiments/polymer_sensitivity_v1/dataset/cases.csv` and `time_series.csv`. The experiment is linked to the domain case `polymer_simple2d` through `analysis_case_id`, while remaining separate from the demo CSV. The analysis is descriptive; it does not establish causality or economic optimality.

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATASET = ROOT / 'outputs/experiments/polymer_sensitivity_v1/dataset'
cases = pd.read_csv(DATASET / 'cases.csv')
series = pd.read_csv(DATASET / 'time_series.csv')

## Data

In [ ]:
quality = pd.DataFrame({
    'dataset': ['cases', 'time_series'],
    'rows': [len(cases), len(series)],
    'columns': [len(cases.columns), len(series.columns)],
    'null_cells': [int(cases.isna().sum().sum()), int(series.isna().sum().sum())],
    'duplicate_keys': [int(cases.case_id.duplicated().sum()), int(series.duplicated(['case_id', 'time_days']).sum())],
})
quality

## Results

In [ ]:
final_pressure = (series.sort_values('time_days').groupby('case_id', as_index=False).tail(1)[['case_id', 'field_pressure_bar']])
metrics = cases.merge(final_pressure, on='case_id', validate='one_to_one')
metrics[['parameter_polymer_concentration', 'parameter_injection_rate', 'final_cumulative_oil_m3', 'final_water_cut_fraction', 'field_pressure_bar']]

In [ ]:
rate_effect = metrics.groupby('parameter_injection_rate')[['final_cumulative_oil_m3', 'final_water_cut_fraction', 'field_pressure_bar']].mean()
concentration_effect = metrics.groupby('parameter_polymer_concentration')[['final_cumulative_oil_m3', 'final_water_cut_fraction', 'field_pressure_bar']].mean()
rate_effect, concentration_effect

## Takeaways

Injection rate is the dominant driver of cumulative oil, final water cut, and final pressure in this grid. Polymer concentration has a much smaller response over 0.5–1.5 kg/m³. Interpret the maximum-oil case as a simulated production outcome, not an economic optimum.